# ❤️ Heart Disease Prediction
### End-to-End ML Pipeline — Fixed & Cleaned

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
data = pd.read_csv('heart.csv')
print(data.shape)
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
print("Duplicates:", data.duplicated().sum())
print("\nMissing values:\n", data.isnull().sum())

## 2. EDA

In [ ]:
data['HeartDisease'].value_counts().plot(kind='bar', color=['steelblue','salmon'])
plt.title('Target Distribution')
plt.xlabel('Heart Disease')
plt.ylabel('Count')
plt.xticks([0,1], ['No Disease','Disease'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), ['Age','RestingBP','Cholesterol','MaxHR']):
    sns.histplot(data[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()
print("Note: Zeros in Cholesterol/RestingBP are likely missing values — will be fixed next.")

In [ ]:
sns.countplot(x=data['Sex'], hue=data['HeartDisease'])
plt.title('Sex vs Heart Disease')
plt.show()

In [ ]:
sns.countplot(x=data['ChestPainType'], hue=data['HeartDisease'])
plt.title('Chest Pain Type vs Heart Disease')
plt.show()

In [ ]:
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

> **Fix**: Zeros in  and  are physiologically impossible — replaced with column mean.

In [ ]:
# Fix impossible zero values
ch_mean = data.loc[data['Cholesterol'] != 0, 'Cholesterol'].mean()
data['Cholesterol'] = data['Cholesterol'].replace(0, ch_mean).round(2)

bp_mean = data.loc[data['RestingBP'] != 0, 'RestingBP'].mean()
data['RestingBP'] = data['RestingBP'].replace(0, bp_mean).round(2)

print("Zeros fixed!")

In [ ]:
# One-hot encode categorical columns
# drop_first=True removes one dummy per category to avoid multicollinearity
data_encode = pd.get_dummies(data, drop_first=True).astype(int)
print("Encoded shape:", data_encode.shape)
data_encode.head()

## 4. Train/Test Split & Scaling

> **Fix**: Scaling is applied ONLY after the train/test split, and ONLY fitted on training data.
> This prevents data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data_encode.drop('HeartDisease', axis=1)
y = data_encode['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit scaler on training data ONLY, then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## 5. Model Training & Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score

models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "SVM (RBF)": SVC(probability=True),
    "Decision Tree": DecisionTreeClassifier(),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': round(acc*100, 2), 'F1 Score': round(f1, 4)})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(results_df.to_string(index=False))

In [ ]:
# Visualise model comparison
results_df.plot(x='Model', y=['Accuracy','F1 Score'], kind='bar', figsize=(10,5))
plt.title('Model Comparison')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 6. Save Best Model (KNN) + Artifacts

In [ ]:
import joblib

# Save KNN model (used by app.py)
joblib.dump(models['KNN'], 'KNN_heart_disease_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X.columns.tolist(), 'columns.pkl')

print("Saved: KNN_heart_disease_model.pkl, scaler.pkl, columns.pkl")
print("Columns:", X.columns.tolist())

## 7. Verify Inference Pipeline

> Test that the saved model + scaler predict correctly on raw inputs (simulating what app.py does).

In [ ]:
# Load saved artifacts (simulate app.py)
loaded_model = joblib.load('KNN_heart_disease_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
loaded_cols = joblib.load('columns.pkl')

test_cases = [
    {'label': 'HIGH RISK: 63M ASY BP=160 Chol=340',
     'input': {'Age':[63],'RestingBP':[160],'Cholesterol':[340],'FastingBS':[1],'MaxHR':[100],'Oldpeak':[3.5],
               'Sex_M':[1],'ChestPainType_ATA':[0],'ChestPainType_NAP':[0],'ChestPainType_TA':[0],
               'RestingECG_Normal':[1],'RestingECG_ST':[0],'ExerciseAngina_Y':[1],'ST_Slope_Flat':[1],'ST_Slope_Up':[0]}},
    {'label': 'LOW RISK: 35F ATA BP=120 Chol=190',
     'input': {'Age':[35],'RestingBP':[120],'Cholesterol':[190],'FastingBS':[0],'MaxHR':[175],'Oldpeak':[0.0],
               'Sex_M':[0],'ChestPainType_ATA':[1],'ChestPainType_NAP':[0],'ChestPainType_TA':[0],
               'RestingECG_Normal':[1],'RestingECG_ST':[0],'ExerciseAngina_Y':[0],'ST_Slope_Flat':[0],'ST_Slope_Up':[1]}},
]

for tc in test_cases:
    df = pd.DataFrame(tc['input'])[loaded_cols]
    scaled = loaded_scaler.transform(df)
    pred = loaded_model.predict(scaled)[0]
    prob = loaded_model.predict_proba(scaled)[0]
    print(f"{tc['label']} => {'HIGH RISK ❤️' if pred==1 else 'LOW RISK ✅'} (confidence: {max(prob)*100:.0f}%)")